# OpLog Anatomy

Audits per-pass layer records, including tensor, graph, module, timing, and grad metadata.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## OpLog Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.OpLog",
    "tl.OpLog.DEFAULT_FILL_STATE",
    "tl.OpLog.FIELD_FORK_POLICY",
    "tl.OpLog.PORTABLE_STATE_SPEC",
    "tl.OpLog._construction_done",
    "tl.OpLog._tracing_finished",
    "tl.OpLog.out",
    "tl.OpLog.out_postfunc",
    "tl.OpLog.out_transform",
    "tl.OpLog.has_saved_args",
    "tl.OpLog.autograd_saved_memory",
    "tl.OpLog.num_autograd_saved_tensors",
    "tl.OpLog.terminal_conditional_id",
    "tl.OpLog.conditional_context_kind",
    "tl.OpLog.is_terminal_conditional_bool",
    "tl.OpLog.conditional_wrapper_kind",
    "tl.OpLog.buffer_address",
    "tl.OpLog.buffer_parent",
    "tl.OpLog.buffer_pass",
    "tl.OpLog.bytes_delta_at_call",
    "tl.OpLog.bytes_peak_at_call",
    "tl.OpLog.args_template",
    "tl.OpLog.saved_args",
    "tl.OpLog.kwargs_template",
    "tl.OpLog.saved_kwargs",
    "tl.OpLog.children",
    "tl.OpLog.output_versions_per_child",
    "tl.OpLog.co_parents",
    "tl.OpLog.conditional_arm_children",
    "tl.OpLog.conditional_elif_children",
    "tl.OpLog.conditional_else_children",
    "tl.OpLog.conditional_entry_children",
    "tl.OpLog.conditional_then_children",
    "tl.OpLog.conditional_branch_depth",
    "tl.OpLog.conditional_branch_stack",
    "tl.OpLog.container_spec",
    "tl.OpLog.module",
    "tl.OpLog.modules",
    "tl.OpLog.copy",
    "tl.OpLog.grad_fn_log",
    "tl.OpLog.capture_index",
    "tl.OpLog.detach_saved_activations",
    "tl.OpLog.edge_uses",
    "tl.OpLog.equivalent_ops",
    "tl.OpLog.annotations",
    "tl.OpLog.is_output_parent",
    "tl.OpLog.flops_backward",
    "tl.OpLog.flops_forward",
    "tl.OpLog.func",
    "tl.OpLog.arg_names",
    "tl.OpLog.func_autocast_state",
    "tl.OpLog.func_call_id",
    "tl.OpLog.code_context",
    "tl.OpLog.func_config",
    "tl.OpLog.is_inplace",
    "tl.OpLog.non_tensor_kwargs",
    "tl.OpLog.func_name",
    "tl.OpLog.func_non_tensor_args",
    "tl.OpLog.non_tensor_pos_args",
    "tl.OpLog.func_rng_states",
    "tl.OpLog.func_duration",
    "tl.OpLog.get_children",
    "tl.OpLog.get_parents",
    "tl.OpLog.grad_dtype",
    "tl.OpLog.grad_fn_id",
    "tl.OpLog.grad_fn_name",
    "tl.OpLog.grad_fn",
    "tl.OpLog.grad_memory",
    "tl.OpLog.grad_memory_str",
    "tl.OpLog.grad_shape",
    "tl.OpLog.grad",
    "tl.OpLog.has_output_variations",
    "tl.OpLog.has_children",
    "tl.OpLog.has_co_parents",
    "tl.OpLog.has_grad",
    "tl.OpLog.has_input_ancestor",
    "tl.OpLog.has_internal_source_ancestor",
    "tl.OpLog.has_parents",
    "tl.OpLog.has_saved_outs",
    "tl.OpLog.has_siblings",
    "tl.OpLog.is_in_conditional_body",
    "tl.OpLog.input_ancestors",
    "tl.OpLog.internal_source_ancestors",
    "tl.OpLog.internal_source_parents",
    "tl.OpLog.interventions",
    "tl.OpLog.io_role",
    "tl.OpLog.is_buffer",
    "tl.OpLog.in_submodule",
    "tl.OpLog.is_final_output",
    "tl.OpLog.is_input",
    "tl.OpLog.is_internal_source",
    "tl.OpLog.is_internal_sink",
    "tl.OpLog.is_atomic_module_op",
    "tl.OpLog.has_output_descendant",
    "tl.OpLog.is_output",
    "tl.OpLog.is_part_of_iterable_output",
    "tl.OpLog.is_scalar_bool",
    "tl.OpLog.is_submodule_input",
    "tl.OpLog.is_submodule_output",
    "tl.OpLog.is_terminal_bool",
    "tl.OpLog.multi_output_index",
    "tl.OpLog.layer_label",
    "tl.OpLog.layer_label_no_pass",
    "tl.OpLog.layer_label_no_pass_short",
    "tl.OpLog._layer_label_raw",
    "tl.OpLog.layer_label_short",
    "tl.OpLog.layer_label_w_pass",
    "tl.OpLog.layer_label_w_pass_short",
    "tl.OpLog.trace_index",
    "tl.OpLog.layer_type",
    "tl.OpLog.type_index",
    "tl.OpLog.atomic_module_call",
    "tl.OpLog.log_tensor_grad",
    "tl.OpLog.lookup_keys",
    "tl.OpLog.macs_backward",
    "tl.OpLog.macs_forward",
    "tl.OpLog.materialize_out",
    "tl.OpLog.materialize_grad",
    "tl.OpLog.max_distance_from_input",
    "tl.OpLog.max_distance_from_output",
    "tl.OpLog.min_distance_from_input",
    "tl.OpLog.min_distance_from_output",
    "tl.OpLog._address_normalized",
    "tl.OpLog._module_boundary_thread_output",
    "tl.OpLog._module_boundary_threads_inputs",
    "tl.OpLog.module_call_depth",
    "tl.OpLog.module_ops_entered",
    "tl.OpLog.output_of_module_calls",
    "tl.OpLog.modules_entered",
    "tl.OpLog.module_entry_argnames",
    "tl.OpLog.output_of_modules",
    "tl.OpLog.num_args_total",
    "tl.OpLog.num_kwargs",
    "tl.OpLog.num_param_tensors",
    "tl.OpLog.num_params_frozen",
    "tl.OpLog.num_params",
    "tl.OpLog.num_params_trainable",
    "tl.OpLog.num_calls",
    "tl.OpLog.num_pos_args",
    "tl.OpLog.equivalence_class",
    "tl.OpLog.compute_index",
    "tl.OpLog.output_descendants",
    "tl.OpLog.output_device",
    "tl.OpLog.container_path",
    "tl.OpLog.params",
    "tl.OpLog.param_memory",
    "tl.OpLog.param_memory_str",
    "tl.OpLog.parent_arg_positions",
    "tl.OpLog.parents",
    "tl.OpLog._param_barcodes",
    "tl.OpLog._param_logs",
    "tl.OpLog.parent_param_ops",
    "tl.OpLog.param_shapes",
    "tl.OpLog.parent_params",
    "tl.OpLog.call_index",
    "tl.OpLog.ops",
    "tl.OpLog.recurrent_ops",
    "tl.OpLog.root_ancestors",
    "tl.OpLog.save_grads",
    "tl.OpLog.save_activation",
    "tl.OpLog.bool_value",
    "tl.OpLog.show",
    "tl.OpLog.siblings",
    "tl.OpLog.source_trace",
    "tl.OpLog.tensor",
    "tl.OpLog.dtype",
    "tl.OpLog._label_raw",
    "tl.OpLog.memory",
    "tl.OpLog.memory_str",
    "tl.OpLog.shape",
    "tl.OpLog.transformed_out",
    "tl.OpLog.transformed_out_dtype",
    "tl.OpLog.transformed_out_memory",
    "tl.OpLog.transformed_out_shape",
    "tl.OpLog.transformed_grad",
    "tl.OpLog.transformed_grad_dtype",
    "tl.OpLog.transformed_grad_memory",
    "tl.OpLog.transformed_grad_shape",
    "tl.OpLog.uses_params",
]

pretty_print_fields(
    layer_pass,
    [
        "layer_label",
        "layer_label_w_pass",
        "call_index",
        "func_name",
        "shape",
        "dtype",
        "has_grad",
        "flops_forward",
        "macs_forward",
    ],
)
print("copy type", type(layer_pass.copy()).__name__)

## Identity and tensor data

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog",
    "tl.OpLog.DEFAULT_FILL_STATE",
    "tl.OpLog.FIELD_FORK_POLICY",
    "tl.OpLog.PORTABLE_STATE_SPEC",
    "tl.OpLog._construction_done",
    "tl.OpLog._tracing_finished",
    "tl.OpLog.out",
    "tl.OpLog.out_postfunc",
    "tl.OpLog.out_transform",
    "tl.OpLog.has_saved_args",
    "tl.OpLog.autograd_saved_memory",
    "tl.OpLog.num_autograd_saved_tensors",
    "tl.OpLog.terminal_conditional_id",
    "tl.OpLog.conditional_context_kind",
    "tl.OpLog.is_terminal_conditional_bool",
    "tl.OpLog.conditional_wrapper_kind",
    "tl.OpLog.buffer_address",
    "tl.OpLog.buffer_parent",
    "tl.OpLog.buffer_pass",
    "tl.OpLog.bytes_delta_at_call",
    "tl.OpLog.bytes_peak_at_call",
    "tl.OpLog.args_template",
    "tl.OpLog.saved_args",
    "tl.OpLog.kwargs_template",
    "tl.OpLog.saved_kwargs",
    "tl.OpLog.children",
]

audit_touch_items("Identity and tensor data", ITEMS, AUDIT_CONTEXT)

## Function metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.output_versions_per_child",
    "tl.OpLog.co_parents",
    "tl.OpLog.conditional_arm_children",
    "tl.OpLog.conditional_elif_children",
    "tl.OpLog.conditional_else_children",
    "tl.OpLog.conditional_entry_children",
    "tl.OpLog.conditional_then_children",
    "tl.OpLog.conditional_branch_depth",
    "tl.OpLog.conditional_branch_stack",
    "tl.OpLog.container_spec",
    "tl.OpLog.module",
    "tl.OpLog.modules",
    "tl.OpLog.copy",
    "tl.OpLog.grad_fn_log",
    "tl.OpLog.capture_index",
    "tl.OpLog.detach_saved_activations",
    "tl.OpLog.edge_uses",
    "tl.OpLog.equivalent_ops",
    "tl.OpLog.annotations",
    "tl.OpLog.is_output_parent",
    "tl.OpLog.flops_backward",
    "tl.OpLog.flops_forward",
    "tl.OpLog.func",
    "tl.OpLog.arg_names",
    "tl.OpLog.func_autocast_state",
    "tl.OpLog.func_call_id",
]

audit_touch_items("Function metadata", ITEMS, AUDIT_CONTEXT)

## Autograd and grads

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.code_context",
    "tl.OpLog.func_config",
    "tl.OpLog.is_inplace",
    "tl.OpLog.non_tensor_kwargs",
    "tl.OpLog.func_name",
    "tl.OpLog.func_non_tensor_args",
    "tl.OpLog.non_tensor_pos_args",
    "tl.OpLog.func_rng_states",
    "tl.OpLog.func_duration",
    "tl.OpLog.get_children",
    "tl.OpLog.get_parents",
    "tl.OpLog.grad_dtype",
    "tl.OpLog.grad_fn_id",
    "tl.OpLog.grad_fn_name",
    "tl.OpLog.grad_fn",
    "tl.OpLog.grad_memory",
    "tl.OpLog.grad_memory_str",
    "tl.OpLog.grad_shape",
    "tl.OpLog.grad",
    "tl.OpLog.has_output_variations",
    "tl.OpLog.has_children",
    "tl.OpLog.has_co_parents",
    "tl.OpLog.has_grad",
    "tl.OpLog.has_input_ancestor",
    "tl.OpLog.has_internal_source_ancestor",
    "tl.OpLog.has_parents",
]

audit_touch_items("Autograd and grads", ITEMS, AUDIT_CONTEXT)

## Graph relationships

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.has_saved_outs",
    "tl.OpLog.has_siblings",
    "tl.OpLog.is_in_conditional_body",
    "tl.OpLog.input_ancestors",
    "tl.OpLog.internal_source_ancestors",
    "tl.OpLog.internal_source_parents",
    "tl.OpLog.interventions",
    "tl.OpLog.io_role",
    "tl.OpLog.is_buffer",
    "tl.OpLog.in_submodule",
    "tl.OpLog.is_final_output",
    "tl.OpLog.is_input",
    "tl.OpLog.is_internal_source",
    "tl.OpLog.is_internal_sink",
    "tl.OpLog.is_atomic_module_op",
    "tl.OpLog.has_output_descendant",
    "tl.OpLog.is_output",
    "tl.OpLog.is_part_of_iterable_output",
    "tl.OpLog.is_scalar_bool",
    "tl.OpLog.is_submodule_input",
    "tl.OpLog.is_submodule_output",
    "tl.OpLog.is_terminal_bool",
    "tl.OpLog.multi_output_index",
    "tl.OpLog.layer_label",
    "tl.OpLog.layer_label_no_pass",
    "tl.OpLog.layer_label_no_pass_short",
]

audit_touch_items("Graph relationships", ITEMS, AUDIT_CONTEXT)

## Module metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog._layer_label_raw",
    "tl.OpLog.layer_label_short",
    "tl.OpLog.layer_label_w_pass",
    "tl.OpLog.layer_label_w_pass_short",
    "tl.OpLog.trace_index",
    "tl.OpLog.layer_type",
    "tl.OpLog.type_index",
    "tl.OpLog.atomic_module_call",
    "tl.OpLog.log_tensor_grad",
    "tl.OpLog.lookup_keys",
    "tl.OpLog.macs_backward",
    "tl.OpLog.macs_forward",
    "tl.OpLog.materialize_out",
    "tl.OpLog.materialize_grad",
    "tl.OpLog.max_distance_from_input",
    "tl.OpLog.max_distance_from_output",
    "tl.OpLog.min_distance_from_input",
    "tl.OpLog.min_distance_from_output",
    "tl.OpLog._address_normalized",
    "tl.OpLog._module_boundary_thread_output",
    "tl.OpLog._module_boundary_threads_inputs",
    "tl.OpLog.module_call_depth",
    "tl.OpLog.module_ops_entered",
    "tl.OpLog.output_of_module_calls",
    "tl.OpLog.modules_entered",
    "tl.OpLog.module_entry_argnames",
]

audit_touch_items("Module metadata", ITEMS, AUDIT_CONTEXT)

## Control-flow and buffering

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.output_of_modules",
    "tl.OpLog.num_args_total",
    "tl.OpLog.num_kwargs",
    "tl.OpLog.num_param_tensors",
    "tl.OpLog.num_params_frozen",
    "tl.OpLog.num_params",
    "tl.OpLog.num_params_trainable",
    "tl.OpLog.num_calls",
    "tl.OpLog.num_pos_args",
    "tl.OpLog.equivalence_class",
    "tl.OpLog.compute_index",
    "tl.OpLog.output_descendants",
    "tl.OpLog.output_device",
    "tl.OpLog.container_path",
    "tl.OpLog.params",
    "tl.OpLog.param_memory",
    "tl.OpLog.param_memory_str",
    "tl.OpLog.parent_arg_positions",
    "tl.OpLog.parents",
    "tl.OpLog._param_barcodes",
    "tl.OpLog._param_logs",
    "tl.OpLog.parent_param_ops",
    "tl.OpLog.param_shapes",
    "tl.OpLog.parent_params",
    "tl.OpLog.call_index",
    "tl.OpLog.ops",
]

audit_touch_items("Control-flow and buffering", ITEMS, AUDIT_CONTEXT)

## Remaining pass fields

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.recurrent_ops",
    "tl.OpLog.root_ancestors",
    "tl.OpLog.save_grads",
    "tl.OpLog.save_activation",
    "tl.OpLog.bool_value",
    "tl.OpLog.show",
    "tl.OpLog.siblings",
    "tl.OpLog.source_trace",
    "tl.OpLog.tensor",
    "tl.OpLog.dtype",
    "tl.OpLog._label_raw",
    "tl.OpLog.memory",
    "tl.OpLog.memory_str",
    "tl.OpLog.shape",
    "tl.OpLog.transformed_out",
    "tl.OpLog.transformed_out_dtype",
    "tl.OpLog.transformed_out_memory",
    "tl.OpLog.transformed_out_shape",
    "tl.OpLog.transformed_grad",
    "tl.OpLog.transformed_grad_dtype",
    "tl.OpLog.transformed_grad_memory",
    "tl.OpLog.transformed_grad_shape",
    "tl.OpLog.uses_params",
]

audit_touch_items("Remaining pass fields", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.OpLog",
    "tl.OpLog.DEFAULT_FILL_STATE",
    "tl.OpLog.FIELD_FORK_POLICY",
    "tl.OpLog.PORTABLE_STATE_SPEC",
    "tl.OpLog._construction_done",
    "tl.OpLog._tracing_finished",
    "tl.OpLog.out",
    "tl.OpLog.out_postfunc",
    "tl.OpLog.out_transform",
    "tl.OpLog.has_saved_args",
    "tl.OpLog.autograd_saved_memory",
    "tl.OpLog.num_autograd_saved_tensors",
    "tl.OpLog.terminal_conditional_id",
    "tl.OpLog.conditional_context_kind",
    "tl.OpLog.is_terminal_conditional_bool",
    "tl.OpLog.conditional_wrapper_kind",
    "tl.OpLog.buffer_address",
    "tl.OpLog.buffer_parent",
    "tl.OpLog.buffer_pass",
    "tl.OpLog.bytes_delta_at_call",
    "tl.OpLog.bytes_peak_at_call",
    "tl.OpLog.args_template",
    "tl.OpLog.saved_args",
    "tl.OpLog.kwargs_template",
    "tl.OpLog.saved_kwargs",
    "tl.OpLog.children",
    "tl.OpLog.output_versions_per_child",
    "tl.OpLog.co_parents",
    "tl.OpLog.conditional_arm_children",
    "tl.OpLog.conditional_elif_children",
    "tl.OpLog.conditional_else_children",
    "tl.OpLog.conditional_entry_children",
    "tl.OpLog.conditional_then_children",
    "tl.OpLog.conditional_branch_depth",
    "tl.OpLog.conditional_branch_stack",
    "tl.OpLog.container_spec",
    "tl.OpLog.module",
    "tl.OpLog.modules",
    "tl.OpLog.copy",
    "tl.OpLog.grad_fn_log",
    "tl.OpLog.capture_index",
    "tl.OpLog.detach_saved_activations",
    "tl.OpLog.edge_uses",
    "tl.OpLog.equivalent_ops",
    "tl.OpLog.annotations",
    "tl.OpLog.is_output_parent",
    "tl.OpLog.flops_backward",
    "tl.OpLog.flops_forward",
    "tl.OpLog.func",
    "tl.OpLog.arg_names",
    "tl.OpLog.func_autocast_state",
    "tl.OpLog.func_call_id",
    "tl.OpLog.code_context",
    "tl.OpLog.func_config",
    "tl.OpLog.is_inplace",
    "tl.OpLog.non_tensor_kwargs",
    "tl.OpLog.func_name",
    "tl.OpLog.func_non_tensor_args",
    "tl.OpLog.non_tensor_pos_args",
    "tl.OpLog.func_rng_states",
    "tl.OpLog.func_duration",
    "tl.OpLog.get_children",
    "tl.OpLog.get_parents",
    "tl.OpLog.grad_dtype",
    "tl.OpLog.grad_fn_id",
    "tl.OpLog.grad_fn_name",
    "tl.OpLog.grad_fn",
    "tl.OpLog.grad_memory",
    "tl.OpLog.grad_memory_str",
    "tl.OpLog.grad_shape",
    "tl.OpLog.grad",
    "tl.OpLog.has_output_variations",
    "tl.OpLog.has_children",
    "tl.OpLog.has_co_parents",
    "tl.OpLog.has_grad",
    "tl.OpLog.has_input_ancestor",
    "tl.OpLog.has_internal_source_ancestor",
    "tl.OpLog.has_parents",
    "tl.OpLog.has_saved_outs",
    "tl.OpLog.has_siblings",
    "tl.OpLog.is_in_conditional_body",
    "tl.OpLog.input_ancestors",
    "tl.OpLog.internal_source_ancestors",
    "tl.OpLog.internal_source_parents",
    "tl.OpLog.interventions",
    "tl.OpLog.io_role",
    "tl.OpLog.is_buffer",
    "tl.OpLog.in_submodule",
    "tl.OpLog.is_final_output",
    "tl.OpLog.is_input",
    "tl.OpLog.is_internal_source",
    "tl.OpLog.is_internal_sink",
    "tl.OpLog.is_atomic_module_op",
    "tl.OpLog.has_output_descendant",
    "tl.OpLog.is_output",
    "tl.OpLog.is_part_of_iterable_output",
    "tl.OpLog.is_scalar_bool",
    "tl.OpLog.is_submodule_input",
    "tl.OpLog.is_submodule_output",
    "tl.OpLog.is_terminal_bool",
    "tl.OpLog.multi_output_index",
    "tl.OpLog.layer_label",
    "tl.OpLog.layer_label_no_pass",
    "tl.OpLog.layer_label_no_pass_short",
    "tl.OpLog._layer_label_raw",
    "tl.OpLog.layer_label_short",
    "tl.OpLog.layer_label_w_pass",
    "tl.OpLog.layer_label_w_pass_short",
    "tl.OpLog.trace_index",
    "tl.OpLog.layer_type",
    "tl.OpLog.type_index",
    "tl.OpLog.atomic_module_call",
    "tl.OpLog.log_tensor_grad",
    "tl.OpLog.lookup_keys",
    "tl.OpLog.macs_backward",
    "tl.OpLog.macs_forward",
    "tl.OpLog.materialize_out",
    "tl.OpLog.materialize_grad",
    "tl.OpLog.max_distance_from_input",
    "tl.OpLog.max_distance_from_output",
    "tl.OpLog.min_distance_from_input",
    "tl.OpLog.min_distance_from_output",
    "tl.OpLog._address_normalized",
    "tl.OpLog._module_boundary_thread_output",
    "tl.OpLog._module_boundary_threads_inputs",
    "tl.OpLog.module_call_depth",
    "tl.OpLog.module_ops_entered",
    "tl.OpLog.output_of_module_calls",
    "tl.OpLog.modules_entered",
    "tl.OpLog.module_entry_argnames",
    "tl.OpLog.output_of_modules",
    "tl.OpLog.num_args_total",
    "tl.OpLog.num_kwargs",
    "tl.OpLog.num_param_tensors",
    "tl.OpLog.num_params_frozen",
    "tl.OpLog.num_params",
    "tl.OpLog.num_params_trainable",
    "tl.OpLog.num_calls",
    "tl.OpLog.num_pos_args",
    "tl.OpLog.equivalence_class",
    "tl.OpLog.compute_index",
    "tl.OpLog.output_descendants",
    "tl.OpLog.output_device",
    "tl.OpLog.container_path",
    "tl.OpLog.params",
    "tl.OpLog.param_memory",
    "tl.OpLog.param_memory_str",
    "tl.OpLog.parent_arg_positions",
    "tl.OpLog.parents",
    "tl.OpLog._param_barcodes",
    "tl.OpLog._param_logs",
    "tl.OpLog.parent_param_ops",
    "tl.OpLog.param_shapes",
    "tl.OpLog.parent_params",
    "tl.OpLog.call_index",
    "tl.OpLog.ops",
    "tl.OpLog.recurrent_ops",
    "tl.OpLog.root_ancestors",
    "tl.OpLog.save_grads",
    "tl.OpLog.save_activation",
    "tl.OpLog.bool_value",
    "tl.OpLog.show",
    "tl.OpLog.siblings",
    "tl.OpLog.source_trace",
    "tl.OpLog.tensor",
    "tl.OpLog.dtype",
    "tl.OpLog._label_raw",
    "tl.OpLog.memory",
    "tl.OpLog.memory_str",
    "tl.OpLog.shape",
    "tl.OpLog.transformed_out",
    "tl.OpLog.transformed_out_dtype",
    "tl.OpLog.transformed_out_memory",
    "tl.OpLog.transformed_out_shape",
    "tl.OpLog.transformed_grad",
    "tl.OpLog.transformed_grad_dtype",
    "tl.OpLog.transformed_grad_memory",
    "tl.OpLog.transformed_grad_shape",
    "tl.OpLog.uses_params",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")